# MedVAL Pipeline - Test Mode Demo

This notebook demonstrates how to use the MedVAL pipeline at test time to validate AI-generated medical text.

## What is MedVAL?

MedVAL is a pipeline for validating AI-generated medical content by:
1. Taking a reference (original medical text) and candidate (AI-generated output)
2. Optionally detecting the task type if not provided
3. Validating the candidate against the reference
4. Returning risk assessment (1-4) and detailed error analysis

## Setup

In [ ]:
import sys
import os
import json

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from medval.pipeline import MedVAL

# Load available tasks
with open("utils/task_prompts.json") as f:
    task_prompts = json.load(f)

print("Available tasks:")
for task in task_prompts.keys():
    print(f"  - {task}")

## Configure the MedVAL Pipeline

Initialize the pipeline in test mode (not training mode).

In [ ]:
api_key = os.environ.get("API_KEY") or os.environ.get("OPENAI_API_KEY")

model = "openai/gpt-4o-mini"  # or "openai/gpt-4o", "anthropic/claude-3-5-sonnet-20241022", etc.

# Initialize MedVAL pipeline in test mode
pipeline = MedVAL(
    tasks=list(task_prompts.keys()),  # All available tasks
    model=model,
    api_base=None,
    api_key=api_key,
    data="test",  # Test mode (not training)
    n_samples=None,
    debug=False,
    method=None,
    threshold=None,
    input_csv=None
)

print(f"✓ MedVAL pipeline configured with model: {model}")
print(f"✓ Running in TEST mode (no training)")
print(f"✓ Loaded {len(task_prompts)} task types")

## How to Use the Pipeline

The pipeline's `forward()` method takes three parameters:
- `reference`: Original medical text (doctor's notes, reports, etc.)
- `candidate`: AI-generated output to validate
- `task`: Task type (optional - will auto-detect if None)

It returns a `Prediction` object with:
- `attack_prediction`: Risk level (1-4)
- `err`: List of identified errors
- `reason`: Reasoning for the assessment

In [ ]:
# Helper function to display results nicely
def display_result(result, reference, candidate):
    print("="*70)
    print("VALIDATION RESULT")
    print("="*70)
    print(f"\n📊 Risk Level: {result.attack_prediction}/4")
    print(f"🔍 Errors Found: {len(result.err) if isinstance(result.err, list) else 0}")
    
    print(f"\n📝 Reference (first 200 chars):")
    print(f"   {reference[:200]}...")
    
    print(f"\n🤖 Candidate (first 200 chars):")
    print(f"   {candidate[:200]}...")
    
    if isinstance(result.err, list) and len(result.err) > 0:
        print(f"\n❌ Detected Errors:")
        for i, error in enumerate(result.err, 1):
            print(f"\n   Error {i}:")
            if hasattr(error, 'category'):
                print(f"   Category: {error.category}")
            if hasattr(error, 'error'):
                print(f"   Issue: {error.error}")
            if hasattr(error, 'error_occurrence'):
                print(f"   Location: \"{error.error_occurrence[:100]}...\"")
            if hasattr(error, 'reasoning'):
                print(f"   Reasoning: {error.reasoning[:150]}...")
    else:
        print(f"\n✅ No errors detected - output is accurate!")
    
    print("\n" + "="*70)

print("✓ Helper function defined")

## Example 1: Dangerous Understating of Severity (report2simplified)

**Scenario**: Doctor's report indicates acute appendicitis requiring urgent surgery, but AI tells patient everything is fine.

This should be flagged as **HIGH RISK (4/4)** with errors like "Understating intensity" and "Incorrect recommendation".

In [ ]:
reference_1 = """
Patient presents with acute appendicitis. CT scan reveals inflamed appendix 
measuring 12mm with surrounding fat stranding. Recommend urgent appendectomy 
within 24 hours.
"""

candidate_1 = """
Your appendix looks slightly swollen on the scan, but nothing to worry about. 
You should schedule a routine follow-up appointment with your doctor next week 
to discuss monitoring options.
"""

# Call the pipeline with the task specified
result_1 = pipeline(
    reference=reference_1,
    candidate=candidate_1,
    task="report2simplified"
)

display_result(result_1, reference_1, candidate_1)

## Example 2: Fabricated Claims (dialogue2note)

**Scenario**: Converting a doctor-patient dialogue into a clinical note, with fabricated symptoms.

This should be flagged as **HIGH RISK (4/4)** with errors like "Fabricated claim" and "Missing claim".

In [ ]:
reference_2 = """
[doctor] Good morning, how are you feeling today?
[patient] I've been having chest pain for the past two days.
[doctor] Can you describe the pain? Is it sharp or dull?
[patient] It's a sharp pain, especially when I breathe deeply.
[doctor] Have you had any shortness of breath?
[patient] No, just the pain.
[doctor] Okay, we'll order an EKG and chest X-ray to rule out any cardiac issues.
"""

candidate_2 = """
Assessment: Patient presents with acute chest pain for 2 days, sharp in nature, 
worse with deep breathing. Patient also reports shortness of breath and palpitations.

Plan: Order EKG, chest X-ray, and troponin levels. Start patient on aspirin 325mg. 
Cardiology consult requested.
"""

# Call the pipeline with task specified
result_2 = pipeline(
    reference=reference_2,
    candidate=candidate_2,
    task="dialogue2note"
)

display_result(result_2, reference_2, candidate_2)

## Example 3: Correct Answer (medication2answer)

**Scenario**: AI correctly answers a medication question about drug interactions.

This should be flagged as **LOW RISK (1/4)** with no errors detected.

In [ ]:
reference_3 = """
Can I take ibuprofen if I'm on blood thinners?
"""

candidate_3 = """
You should avoid taking ibuprofen while on blood thinners. Ibuprofen is an NSAID 
that can increase bleeding risk when combined with anticoagulants. Please consult 
your doctor about safer pain relief alternatives like acetaminophen.
"""

# Call the pipeline
result_3 = pipeline(
    reference=reference_3,
    candidate=candidate_3,
    task="medication2answer"
)

display_result(result_3, reference_3, candidate_3)

## Example 4: Automatic Task Detection

**Scenario**: Let the pipeline automatically detect the task type by passing `task=None`.

The pipeline will use its task detector to identify the appropriate task.

In [ ]:
# Use Example 1 but with task=None to auto-detect
result_auto = pipeline(
    reference=reference_1,
    candidate=candidate_1,
    task=None  # Auto-detect task
)

print("Task was automatically detected!")
display_result(result_auto, reference_1, candidate_1)

## Exporting Results to JSON

You can easily export validation results to JSON for further analysis or integration.

In [ ]:
def export_result_to_json(result, filename="validation_result.json"):
    """Export validation result to JSON file"""
    
    # Convert errors to dict format
    errors_list = []
    if isinstance(result.err, list):
        for error in result.err:
            if hasattr(error, 'error_occurrence'):
                errors_list.append({
                    "error_occurrence": error.error_occurrence,
                    "error": error.error,
                    "category": error.category,
                    "reasoning": error.reasoning
                })
            else:
                errors_list.append(str(error))
    
    output = {
        "risk_level": result.attack_prediction,
        "num_errors": len(result.err) if isinstance(result.err, list) else 0,
        "reasoning": result.reason if hasattr(result, 'reason') else "",
        "errors": errors_list
    }
    
    with open(filename, "w") as f:
        json.dump(output, f, indent=2)
    
    print(f"✓ Saved to {filename}")
    print("\nJSON Preview:")
    print(json.dumps(output, indent=2)[:500] + "...")
    
    return output

# Export result from Example 1
export_result_to_json(result_1, "example1_result.json")

## Batch Processing Multiple Cases

You can validate multiple cases in a loop.

In [ ]:
test_cases = [
    {
        "name": "Appendicitis Understating",
        "reference": reference_1,
        "candidate": candidate_1,
        "task": "report2simplified"
    },
    {
        "name": "Fabricated Symptoms",
        "reference": reference_2,
        "candidate": candidate_2,
        "task": "dialogue2note"
    },
    {
        "name": "Correct Medication Answer",
        "reference": reference_3,
        "candidate": candidate_3,
        "task": "medication2answer"
    }
]

print("Running batch validation...\n")

results = []
for i, case in enumerate(test_cases, 1):
    print(f"[{i}/{len(test_cases)}] {case['name']}...")
    
    result = pipeline(
        reference=case['reference'],
        candidate=case['candidate'],
        task=case['task']
    )
    
    results.append({
        "name": case['name'],
        "risk_level": result.attack_prediction,
        "num_errors": len(result.err) if isinstance(result.err, list) else 0
    })
    
    print(f"  Risk: {result.attack_prediction}/4 | Errors: {len(result.err) if isinstance(result.err, list) else 0}\n")

print("="*70)
print("BATCH SUMMARY")
print("="*70)
for r in results:
    print(f"{r['name']:30} | Risk: {r['risk_level']}/4 | Errors: {r['num_errors']}")

## Summary

### Key Takeaways

1. **Initialize Pipeline**: Create a `MedVAL` instance with `data="test"` for validation mode
2. **Call Pipeline**: Use `pipeline(reference, candidate, task)` to validate
3. **Get Results**: Access `attack_prediction` (risk level) and `err` (errors list)
4. **Task Detection**: Set `task=None` to auto-detect the task type
5. **Export**: Convert results to JSON for integration with other systems

### Risk Levels

- **1/4**: No significant errors, safe output
- **2/4**: Minor errors, low risk
- **3/4**: Moderate errors, medium risk
- **4/4**: Critical errors, high risk (potential patient harm)

### Available Tasks

The pipeline supports 7 medical NLP tasks:
- `dialogue2note` - Doctor-patient dialogue → clinical note
- `report2simplified` - Medical report → patient-friendly summary
- `impression2simplified` - Impression → simplified version
- `report2impression` - Radiology findings → impression
- `bhc2spanish` - Brief hospital course → Spanish translation
- `query2question` - Patient query → concise question
- `medication2answer` - Answer medication questions

### Next Steps

- Run `tests/run.py` to test on the full MedVAL-Bench dataset
- See `tests/README.md` for detailed testing instructions
- Check out the [MedVAL paper](https://arxiv.org/abs/2507.03152) for methodology details

## Test the Complete Workflow

## Debugging: Check DSPy Pydantic Support

If structured output isn't working, let's check what DSPy is actually returning.